# Issue #251 — canonical identity coverage and exception evidence

This notebook reads the machine-readable package built by `scripts/build_bj251_canonical_identity.py`. It does not open outcome, payout, or settlement-value columns.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path('results/bj251_canonical_identity_evidence_v1')
manifest = json.loads((root / 'MANIFEST.json').read_text(encoding='utf-8'))
coverage = pd.read_csv(root / 'tables/coverage.csv')
ledger = pd.read_csv(root / 'tables/ambiguity_ledger.csv')
aliases = pd.read_csv(root / 'tables/temporal_aliases.csv')
manifest

{'contract_version': 'BJ-UNIFIED-CANONICAL-IDENTITY-001-v1', 'disposition': 'CANDIDATE_PENDING_PI_ACCEPTANCE', 'exception_count': 20909, 'external_full_package': '/home/drwho1985/workspace/soccer/beijing-lot/data/derived/bj251_canonical_identity_v1', 'external_sha256sums_sha256': '872c4a6ddc99360a87dae25ac49ddb8edd25992f110577c3dadfe357235cca6c', 'fixture_count': 214854, 'trace_count': 236327}

In [2]:
coverage.groupby(['source_id', 'causal_grade'])[['rows', 'valid_fixture_rows', 'ambiguous_rows', 'unmatched_rows']].sum()

                                            rows  ...  unmatched_rows
source_id            causal_grade                 ...                
beidan_history_db    SETTLEMENT_ONLY       38463  ...               0
beidan_live_db       STRICT_CAUSAL          5155  ...               0
beidan_xlsm_workbook RETROSPECTIVE_PROXY  192709  ...            2560

[3 rows x 4 columns]

In [3]:
ledger.groupby(['exception_class', 'disposition']).size().rename('rows').to_frame()

                                                   rows
exception_class                    disposition         
CROSS_SOURCE_ENTITY_ALIAS_CONFLICT MANUAL_REVIEW  18349
MISSING_CANONICAL_SLOT             QUARANTINED     2560

In [4]:
read_boundary = json.loads((root / 'READ_BOUNDARY.json').read_text(encoding='utf-8'))
case_policy = pd.read_csv(root / 'tables/exceptional_case_policy.csv')
print('Outcome-exposed inputs:', read_boundary['outcome_exposed_inputs'])
print('Fuzzy auto-acceptance:', read_boundary['fuzzy_auto_acceptance'])
print('Observed temporal aliases:', len(aliases))
case_policy

Outcome-exposed inputs: False
Fuzzy auto-acceptance: False
Observed temporal aliases: 6785


       case_class  ...                                               rule
0     RESCHEDULED  ...  retain issue slot but require provenance-bound...
1       POSTPONED  ...  do not auto-map until a source declares a repl...
2       ABANDONED  ...  retain source trace; never create a settlement...
3       DUPLICATE  ...  multiple source rows for one source and exact ...
4   NEUTRAL_VENUE  ...  venue does not alter home/away team identity; ...
5  RENAMED_ENTITY  ...  only a provenance-bound temporal alias may con...

[6 rows x 3 columns]